# VGG16 — Entrenamiento

In [ ]:
# CELDA 1 — Montar Drive y descomprimir
from google.colab import drive
drive.mount('/content/drive')

import subprocess
result = subprocess.run([
    'unzip', '-q',
    '/content/drive/MyDrive/Datasets/Dataset_P2_DL.zip',
    '-d', '/content/data'
])
print('Descompresión lista')

Mounted at /content/drive
Descompresión lista


In [ ]:
# CELDA 2 — Limpiar archivos corruptos
from PIL import Image
import os

corrupted = []
for root, _, files in os.walk('/content/data'):
    for fname in files:
        fpath = os.path.join(root, fname)
        try:
            with Image.open(fpath) as img:
                img.verify()
        except Exception:
            corrupted.append(fpath)

print(f'Archivos corruptos: {len(corrupted)}')
for f in corrupted:
    print(f)
    os.remove(f)
print('Eliminados')

Archivos corruptos: 0
Eliminados


### Baseline

In [ ]:
# CELDA 3 — Arquitectura VGG16 adaptada a 224x224
import torch
from torch.nn import Conv2d, ReLU, MaxPool2d, Linear, Module, Flatten

class VGGNet(Module):
    def __init__(self, n_channels, n_classes):
        super(VGGNet, self).__init__()
        # Block 1
        self.conv1_1 = Conv2d(n_channels, 64,  kernel_size=3, padding=1, stride=1)
        self.conv1_2 = Conv2d(64,  64,  kernel_size=3, padding=1, stride=1)
        # Block 2
        self.conv2_1 = Conv2d(64,  128, kernel_size=3, padding=1, stride=1)
        self.conv2_2 = Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        # Block 3
        self.conv3_1 = Conv2d(128, 256, kernel_size=3, padding=1, stride=1)
        self.conv3_2 = Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.conv3_3 = Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        # Block 4
        self.conv4_1 = Conv2d(256, 512, kernel_size=3, padding=1, stride=1)
        self.conv4_2 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.conv4_3 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        # Block 5
        self.conv5_1 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.conv5_2 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.conv5_3 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        # Pooling
        self.pool    = MaxPool2d(kernel_size=2, stride=2)
        # Flatten
        self.flatten = Flatten()
        # Fully connected  (224 → 5 maxpools → 7x7)
        self.fc1 = Linear(512 * 7 * 7, 4096)
        self.fc2 = Linear(4096, 4096)
        self.fc3 = Linear(4096, n_classes)
        # Activation
        self.relu = ReLU(inplace=True)

    def forward(self, x):
        # Block 1
        x = self.relu(self.conv1_1(x))
        x = self.relu(self.conv1_2(x))
        x = self.pool(x)
        # Block 2
        x = self.relu(self.conv2_1(x))
        x = self.relu(self.conv2_2(x))
        x = self.pool(x)
        # Block 3
        x = self.relu(self.conv3_1(x))
        x = self.relu(self.conv3_2(x))
        x = self.relu(self.conv3_3(x))
        x = self.pool(x)
        # Block 4
        x = self.relu(self.conv4_1(x))
        x = self.relu(self.conv4_2(x))
        x = self.relu(self.conv4_3(x))
        x = self.pool(x)
        # Block 5
        x = self.relu(self.conv5_1(x))
        x = self.relu(self.conv5_2(x))
        x = self.relu(self.conv5_3(x))
        x = self.pool(x)
        # Flatten
        x = self.flatten(x)
        # Fully connected
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

print('Arquitectura definida')

Arquitectura definida


In [ ]:
# CELDA 4 — Imports, configuración, datos y loaders
import os, time, json, copy
from pathlib import Path
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# ── Hiperparámetros ─────────────────────────────────────────
SEED             = 42
EPOCHS           = 60
BATCH_SIZE       = 128
LR               = 0.01      # SGD
MOMENTUM         = 0.9
WEIGHT_DECAY     = 5e-4
NUM_WORKERS      = 4
CHECKPOINT_EVERY = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Rutas ───────────────────────────────────────────────────
DATA_PATH      = '/content/data/Data_subsampled'
OUT_DIR        = Path('/content/drive/MyDrive/Datasets/outputs_vgg16_baseline')
CHECKPOINT_DIR = OUT_DIR / 'checkpoints'
PLOTS_DIR      = OUT_DIR / 'plots'
METRICS_DIR    = OUT_DIR / 'metrics'
for p in (CHECKPOINT_DIR, PLOTS_DIR, METRICS_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ── Reproducibilidad ────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ── Media y std del dataset ──────────────────────────────────
def compute_mean_std(data_path):
    ds     = datasets.ImageFolder(data_path, transform=transforms.ToTensor())
    loader = DataLoader(ds, batch_size=128, num_workers=4, shuffle=False,
                        persistent_workers=True)
    mean = torch.zeros(3)
    std  = torch.zeros(3)
    for imgs, _ in loader:
        mean += imgs.mean(dim=[0, 2, 3])
        std  += imgs.std(dim=[0, 2, 3])
    mean /= len(loader)
    std  /= len(loader)
    print(f'Mean: {mean}  Std: {std}')
    return mean.tolist(), std.tolist()

MEAN, STD = compute_mean_std(DATA_PATH)

# ── Transforms ──────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Dataset y splits ────────────────────────────────────────
dataset      = datasets.ImageFolder(root=DATA_PATH, transform=train_transform)
dataset_eval = datasets.ImageFolder(root=DATA_PATH, transform=eval_transform)

class_names = dataset.classes
n_classes   = len(class_names)
print(f'Clases: {class_names}  |  Total: {len(dataset)}  |  Device: {DEVICE}')

targets = np.array(dataset.targets)
indices = np.arange(len(dataset))

train_idx, temp_idx, y_train, y_temp = train_test_split(
    indices, targets, train_size=0.8, stratify=targets, random_state=SEED)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

train_dataset = Subset(dataset,      train_idx)
val_dataset   = Subset(dataset_eval, val_idx)
test_dataset  = Subset(dataset_eval, test_idx)
print(f'train:{len(train_dataset)}  val:{len(val_dataset)}  test:{len(test_dataset)}')

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                 pin_memory=True, persistent_workers=True, prefetch_factor=2)
train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kw)
print('Loaders listos')

Mean: tensor([0.3745, 0.3034, 0.4152])  Std: tensor([0.3549, 0.3974, 0.4199])
Clases: ['Disgust', 'Fear', 'Happy', 'Neutral', 'Sad']  |  Total: 30000  |  Device: cuda
train:24000  val:3000  test:3000
Loaders listos


In [ ]:
# CELDA 5 — Utilidades y funciones de entrenamiento

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def save_checkpoint(state, filename):
    torch.save(state, str(filename))

def save_curves(history, outdir):
    epochs = list(range(1, len(history['train_loss']) + 1))
    for keys, title, fname in [
        (('train_loss','val_loss'), 'Loss',     'loss_curve.png'),
        (('train_f1',  'val_f1'),  'Macro F1', 'f1_curve.png'),
    ]:
        plt.figure()
        for k in keys: plt.plot(epochs, history[k], label=k)
        plt.xlabel('Epoch'); plt.ylabel(title)
        plt.title(f'{title} - Train/Val'); plt.legend(); plt.grid(True)
        plt.savefig(outdir / fname, dpi=150); plt.close()
    import csv
    with open(outdir / 'metrics_per_epoch.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['epoch','train_loss','val_loss','train_f1','val_f1','epoch_time_s'])
        for i in range(len(epochs)):
            w.writerow([epochs[i], history['train_loss'][i], history['val_loss'][i],
                        history['train_f1'][i], history['val_f1'][i], history['epoch_times'][i]])

def save_confusion_matrix(y_true, y_pred, classes, outpath, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    cm_plot = cm.astype('float') / (cm.sum(axis=1)[:, None] + 1e-12) if normalize else cm
    plt.figure(figsize=(8, 6))
    plt.imshow(cm_plot, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de confusión'); plt.colorbar()
    ticks = np.arange(len(classes))
    plt.xticks(ticks, classes, rotation=45, ha='right')
    plt.yticks(ticks, classes)
    fmt = '.2f' if normalize else 'd'
    thresh = cm_plot.max() / 2.
    for i, j in np.ndindex(cm_plot.shape):
        plt.text(j, i, format(cm_plot[i, j], fmt), ha='center',
                 color='white' if cm_plot[i, j] > thresh else 'black')
    plt.ylabel('Clase real'); plt.xlabel('Clase predicha')
    plt.tight_layout(); plt.savefig(outpath, dpi=150); plt.close()
    np.save(str(outpath.with_suffix('.npy')), cm)

def run_epoch(model, loader, criterion, optimizer, device, training=True):
    model.train() if training else model.eval()
    losses, preds_all, tgts_all = [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, tgts in loader:
            imgs, tgts = imgs.to(device), tgts.to(device)
            out  = model(imgs)
            loss = criterion(out, tgts)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            losses.append(loss.item())
            preds_all.extend(out.argmax(1).cpu().numpy())
            tgts_all.extend(tgts.cpu().numpy())
    avg_loss = float(np.mean(losses))
    f1  = float(f1_score(tgts_all, preds_all, average='macro', zero_division=0))
    acc = float(accuracy_score(tgts_all, preds_all))
    return avg_loss, f1, acc, np.array(preds_all), np.array(tgts_all)

def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)

    history = {k: [] for k in ('train_loss','val_loss','train_f1','val_f1','epoch_times')}
    best_val_acc, best_state = -1.0, None

    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()
        tr_loss, tr_f1, tr_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer, DEVICE, training=True)
        vl_loss, vl_f1, vl_acc, vp, vt = run_epoch(model, val_loader, criterion, None, DEVICE, training=False)
        scheduler.step(vl_loss)
        elapsed = time.perf_counter() - t0

        for k, v in zip(('train_loss','val_loss','train_f1','val_f1','epoch_times'),
                        (tr_loss, vl_loss, tr_f1, vl_f1, elapsed)):
            history[k].append(v)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'Ep {epoch:3d}/{EPOCHS} | '
              f'tr_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} tr_acc={tr_acc:.4f} | '
              f'vl_loss={vl_loss:.4f} vl_f1={vl_f1:.4f} vl_acc={vl_acc:.4f} | '
              f'lr={lr_now:.5f} | {elapsed:.1f}s')

        if epoch % CHECKPOINT_EVERY == 0:
            save_checkpoint({'epoch': epoch, 'model_state': model.state_dict(),
                             'optimizer_state': optimizer.state_dict(), 'history': history},
                            CHECKPOINT_DIR / f'checkpoint_epoch_{epoch:03d}.pth')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state = {'epoch': epoch, 'val_acc': vl_acc,
                          'model_state': copy.deepcopy(model.state_dict()),
                          'history': history}
            save_checkpoint(best_state, CHECKPOINT_DIR / 'best_model.pth')
            print(f'  ★ Mejor modelo (val_acc={vl_acc:.4f})')

    save_checkpoint({'epoch': EPOCHS, 'model_state': model.state_dict(), 'history': history},
                    CHECKPOINT_DIR / 'final_model.pth')
    return history, best_state

print('Funciones definidas')

Funciones definidas


In [ ]:
# CELDA 6 — Entrenamiento y evaluación final
model    = VGGNet(n_channels=3, n_classes=n_classes).to(DEVICE)
n_params = count_parameters(model)
print(f'Parámetros: {n_params:,} ({n_params/1e6:.3f} M)')

history, best_state = train_model(model, train_loader, val_loader)

# Guardar curvas e historial
save_curves(history, PLOTS_DIR)
with open(METRICS_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

criterion = nn.CrossEntropyLoss()

# Evaluación en validación
model.load_state_dict(best_state['model_state'])
model.to(DEVICE)
_, vl_f1, vl_acc, vl_preds, vl_tgts = run_epoch(
    model, val_loader, criterion, None, DEVICE, training=False)
print(f'[VAL  best] acc={vl_acc:.4f}  f1={vl_f1:.4f}')
save_confusion_matrix(vl_tgts, vl_preds, class_names, PLOTS_DIR / 'confusion_matrix_val.png')
report = classification_report(vl_tgts, vl_preds, target_names=class_names, digits=4, zero_division=0)
(METRICS_DIR / 'classification_report_val.txt').write_text(report)
np.save(METRICS_DIR / 'val_preds.npy',   vl_preds)
np.save(METRICS_DIR / 'val_targets.npy', vl_tgts)

# Evaluación en test
_, ts_f1, ts_acc, ts_preds, ts_tgts = run_epoch(
    model, test_loader, criterion, None, DEVICE, training=False)
print(f'[TEST best] acc={ts_acc:.4f}  f1={ts_f1:.4f}')
(METRICS_DIR / 'test_metrics.txt').write_text(
    f'test_acc={ts_acc:.4f}\ntest_f1={ts_f1:.4f}\n')
np.save(METRICS_DIR / 'test_preds.npy',   ts_preds)
np.save(METRICS_DIR / 'test_targets.npy', ts_tgts)

# Resumen final
summary = (
    f'n_parameters={n_params}\n'
    f'avg_time_per_epoch={np.mean(history["epoch_times"]):.2f}s\n'
    f'best_val_acc={best_state["val_acc"]:.4f}\n'
    f'best_val_f1={max(history["val_f1"]):.4f}\n'
    f'test_acc={ts_acc:.4f}\n'
    f'test_f1={ts_f1:.4f}\n'
)
(METRICS_DIR / 'final_summary.txt').write_text(summary)
print('\n' + summary)

Parámetros: 134,281,029 (134.281 M)
Ep   1/60 | tr_loss=1.6097 tr_f1=0.1744 tr_acc=0.1959 | vl_loss=1.6095 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 42.6s
  ★ Mejor modelo (val_acc=0.2000)
Ep   2/60 | tr_loss=1.6096 tr_f1=0.1950 tr_acc=0.1983 | vl_loss=1.6095 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 41.0s
Ep   3/60 | tr_loss=1.6096 tr_f1=0.1469 tr_acc=0.2014 | vl_loss=1.6097 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 41.1s
Ep   4/60 | tr_loss=1.6097 tr_f1=0.1720 tr_acc=0.1976 | vl_loss=1.6097 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 41.0s
Ep   5/60 | tr_loss=1.6097 tr_f1=0.1875 tr_acc=0.2003 | vl_loss=1.6096 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 40.9s
Ep   6/60 | tr_loss=1.6097 tr_f1=0.1575 tr_acc=0.1998 | vl_loss=1.6095 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 41.0s
Ep   7/60 | tr_loss=1.6097 tr_f1=0.1900 tr_acc=0.1959 | vl_loss=1.6095 vl_f1=0.0667 vl_acc=0.2000 | lr=0.00500 | 41.0s
Ep   8/60 | tr_loss=1.6095 tr_f1=0.1502 tr_acc=0.1995 | vl_loss=1.6095 vl_f1=0.0667 vl_acc=0.2000

### V1 — VGG16 + BatchNorm

In [ ]:
# CELDA 3 — Arquitectura VGG16 adaptada a 224x224
import torch
from torch.nn import Conv2d, ReLU, MaxPool2d, Linear, Module, Flatten, BatchNorm2d


class VGGNet(Module):
    def __init__(self, n_channels, n_classes):
        super(VGGNet, self).__init__()
        # Block 1
        self.conv1_1 = Conv2d(n_channels, 64, kernel_size=3, padding=1)
        self.bn1_1   = BatchNorm2d(64)   # ← NUEVO
        self.conv1_2 = Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn1_2   = BatchNorm2d(64)   # ← NUEVO
        # Block 2
        self.conv2_1 = Conv2d(64,  128, kernel_size=3, padding=1, stride=1)
        self.bn2_1   = BatchNorm2d(128)   # ← NUEVO
        self.conv2_2 = Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn2_2   = BatchNorm2d(128)   # ← NUEVO
        # Block 3
        self.conv3_1 = Conv2d(128, 256, kernel_size=3, padding=1, stride=1)
        self.bn3_1   = BatchNorm2d(256)   # ← NUEVO
        self.conv3_2 = Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn3_2   = BatchNorm2d(256)   # ← NUEVO
        self.conv3_3 = Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn3_3   = BatchNorm2d(256)   # ← NUEVO
        # Block 4
        self.conv4_1 = Conv2d(256, 512, kernel_size=3, padding=1, stride=1)
        self.bn4_1   = BatchNorm2d(512)   # ← NUEVO
        self.conv4_2 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn4_2   = BatchNorm2d(512)   # ← NUEVO
        self.conv4_3 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn4_3   = BatchNorm2d(512)   # ← NUEVO
        # Block 5
        self.conv5_1 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn5_1   = BatchNorm2d(512)   # ← NUEVO
        self.conv5_2 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn5_2   = BatchNorm2d(512)   # ← NUEVO
        self.conv5_3 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn5_3   = BatchNorm2d(512)   # ← NUEVO
        # Pooling
        self.pool    = MaxPool2d(kernel_size=2, stride=2)
        # Flatten
        self.flatten = Flatten()
        # Fully connected  (224 → 5 maxpools → 7x7)
        self.fc1 = Linear(512 * 7 * 7, 4096)
        self.fc2 = Linear(4096, 4096)
        self.fc3 = Linear(4096, n_classes)
        # Activation
        self.relu = ReLU(inplace=True)

    def forward(self, x):
        # Block 1
        x = self.relu(self.bn1_1(self.conv1_1(x)))  # conv → BN → ReLU
        x = self.relu(self.bn1_2(self.conv1_2(x)))
        x = self.pool(x)
        # Block 2
        x = self.relu(self.bn2_1(self.conv2_1(x)))
        x = self.relu(self.bn2_2(self.conv2_2(x)))
        x = self.pool(x)
        # Block 3
        x = self.relu(self.bn3_1(self.conv3_1(x)))
        x = self.relu(self.bn3_2(self.conv3_2(x)))
        x = self.relu(self.bn3_3(self.conv3_3(x)))
        x = self.pool(x)
        # Block 4
        x = self.relu(self.bn4_1(self.conv4_1(x)))
        x = self.relu(self.bn4_2(self.conv4_2(x)))
        x = self.relu(self.bn4_3(self.conv4_3(x)))
        x = self.pool(x)
        # Block 5
        x = self.relu(self.bn5_1(self.conv5_1(x)))
        x = self.relu(self.bn5_2(self.conv5_2(x)))
        x = self.relu(self.bn5_3(self.conv5_3(x)))
        x = self.pool(x)
        # Flatten
        x = self.flatten(x)
        # Fully connected
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

print('Arquitectura definida')

Arquitectura definida


In [ ]:
# CELDA 4 — Imports, configuración, datos y loaders
import os, time, json, copy
from pathlib import Path
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# ── Hiperparámetros ─────────────────────────────────────────
SEED             = 42
EPOCHS           = 60
BATCH_SIZE       = 128
LR               = 0.01      # SGD
MOMENTUM         = 0.9
WEIGHT_DECAY     = 5e-4
NUM_WORKERS      = 4
CHECKPOINT_EVERY = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Rutas ───────────────────────────────────────────────────
DATA_PATH      = '/content/data/Data_subsampled'
OUT_DIR        = Path('/content/drive/MyDrive/Datasets/outputs_vgg16_v1')
CHECKPOINT_DIR = OUT_DIR / 'checkpoints'
PLOTS_DIR      = OUT_DIR / 'plots'
METRICS_DIR    = OUT_DIR / 'metrics'
for p in (CHECKPOINT_DIR, PLOTS_DIR, METRICS_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ── Reproducibilidad ────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ── Media y std del dataset ──────────────────────────────────
def compute_mean_std(data_path):
    ds     = datasets.ImageFolder(data_path, transform=transforms.ToTensor())
    loader = DataLoader(ds, batch_size=128, num_workers=4, shuffle=False,
                        persistent_workers=True)
    mean = torch.zeros(3)
    std  = torch.zeros(3)
    for imgs, _ in loader:
        mean += imgs.mean(dim=[0, 2, 3])
        std  += imgs.std(dim=[0, 2, 3])
    mean /= len(loader)
    std  /= len(loader)
    print(f'Mean: {mean}  Std: {std}')
    return mean.tolist(), std.tolist()

MEAN, STD = compute_mean_std(DATA_PATH)

# ── Transforms ──────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Dataset y splits ────────────────────────────────────────
dataset      = datasets.ImageFolder(root=DATA_PATH, transform=train_transform)
dataset_eval = datasets.ImageFolder(root=DATA_PATH, transform=eval_transform)

class_names = dataset.classes
n_classes   = len(class_names)
print(f'Clases: {class_names}  |  Total: {len(dataset)}  |  Device: {DEVICE}')

targets = np.array(dataset.targets)
indices = np.arange(len(dataset))

train_idx, temp_idx, y_train, y_temp = train_test_split(
    indices, targets, train_size=0.8, stratify=targets, random_state=SEED)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

train_dataset = Subset(dataset,      train_idx)
val_dataset   = Subset(dataset_eval, val_idx)
test_dataset  = Subset(dataset_eval, test_idx)
print(f'train:{len(train_dataset)}  val:{len(val_dataset)}  test:{len(test_dataset)}')

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                 pin_memory=True, persistent_workers=True, prefetch_factor=2)
train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kw)
print('Loaders listos')

Mean: tensor([0.3745, 0.3034, 0.4152])  Std: tensor([0.3549, 0.3974, 0.4199])
Clases: ['Disgust', 'Fear', 'Happy', 'Neutral', 'Sad']  |  Total: 30000  |  Device: cuda
train:24000  val:3000  test:3000
Loaders listos


In [ ]:
# CELDA 5 — Utilidades y funciones de entrenamiento

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def save_checkpoint(state, filename):
    torch.save(state, str(filename))

def save_curves(history, outdir):
    epochs = list(range(1, len(history['train_loss']) + 1))
    for keys, title, fname in [
        (('train_loss','val_loss'), 'Loss',     'loss_curve.png'),
        (('train_f1',  'val_f1'),  'Macro F1', 'f1_curve.png'),
    ]:
        plt.figure()
        for k in keys: plt.plot(epochs, history[k], label=k)
        plt.xlabel('Epoch'); plt.ylabel(title)
        plt.title(f'{title} - Train/Val'); plt.legend(); plt.grid(True)
        plt.savefig(outdir / fname, dpi=150); plt.close()
    import csv
    with open(outdir / 'metrics_per_epoch.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['epoch','train_loss','val_loss','train_f1','val_f1','epoch_time_s'])
        for i in range(len(epochs)):
            w.writerow([epochs[i], history['train_loss'][i], history['val_loss'][i],
                        history['train_f1'][i], history['val_f1'][i], history['epoch_times'][i]])

def save_confusion_matrix(y_true, y_pred, classes, outpath, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    cm_plot = cm.astype('float') / (cm.sum(axis=1)[:, None] + 1e-12) if normalize else cm
    plt.figure(figsize=(8, 6))
    plt.imshow(cm_plot, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de confusión'); plt.colorbar()
    ticks = np.arange(len(classes))
    plt.xticks(ticks, classes, rotation=45, ha='right')
    plt.yticks(ticks, classes)
    fmt = '.2f' if normalize else 'd'
    thresh = cm_plot.max() / 2.
    for i, j in np.ndindex(cm_plot.shape):
        plt.text(j, i, format(cm_plot[i, j], fmt), ha='center',
                 color='white' if cm_plot[i, j] > thresh else 'black')
    plt.ylabel('Clase real'); plt.xlabel('Clase predicha')
    plt.tight_layout(); plt.savefig(outpath, dpi=150); plt.close()
    np.save(str(outpath.with_suffix('.npy')), cm)

def run_epoch(model, loader, criterion, optimizer, device, training=True):
    model.train() if training else model.eval()
    losses, preds_all, tgts_all = [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, tgts in loader:
            imgs, tgts = imgs.to(device), tgts.to(device)
            out  = model(imgs)
            loss = criterion(out, tgts)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            losses.append(loss.item())
            preds_all.extend(out.argmax(1).cpu().numpy())
            tgts_all.extend(tgts.cpu().numpy())
    avg_loss = float(np.mean(losses))
    f1  = float(f1_score(tgts_all, preds_all, average='macro', zero_division=0))
    acc = float(accuracy_score(tgts_all, preds_all))
    return avg_loss, f1, acc, np.array(preds_all), np.array(tgts_all)

def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)

    history = {k: [] for k in ('train_loss','val_loss','train_f1','val_f1','epoch_times')}
    best_val_acc, best_state = -1.0, None

    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()
        tr_loss, tr_f1, tr_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer, DEVICE, training=True)
        vl_loss, vl_f1, vl_acc, vp, vt = run_epoch(model, val_loader, criterion, None, DEVICE, training=False)
        scheduler.step(vl_loss)
        elapsed = time.perf_counter() - t0

        for k, v in zip(('train_loss','val_loss','train_f1','val_f1','epoch_times'),
                        (tr_loss, vl_loss, tr_f1, vl_f1, elapsed)):
            history[k].append(v)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'Ep {epoch:3d}/{EPOCHS} | '
              f'tr_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} tr_acc={tr_acc:.4f} | '
              f'vl_loss={vl_loss:.4f} vl_f1={vl_f1:.4f} vl_acc={vl_acc:.4f} | '
              f'lr={lr_now:.5f} | {elapsed:.1f}s')

        if epoch % CHECKPOINT_EVERY == 0:
            save_checkpoint({'epoch': epoch, 'model_state': model.state_dict(),
                             'optimizer_state': optimizer.state_dict(), 'history': history},
                            CHECKPOINT_DIR / f'checkpoint_epoch_{epoch:03d}.pth')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state = {'epoch': epoch, 'val_acc': vl_acc,
                          'model_state': copy.deepcopy(model.state_dict()),
                          'history': history}
            save_checkpoint(best_state, CHECKPOINT_DIR / 'best_model.pth')
            print(f'  ★ Mejor modelo (val_acc={vl_acc:.4f})')

    save_checkpoint({'epoch': EPOCHS, 'model_state': model.state_dict(), 'history': history},
                    CHECKPOINT_DIR / 'final_model.pth')
    return history, best_state

print('Funciones definidas')

Funciones definidas


In [ ]:
# CELDA 6 — Entrenamiento y evaluación final
model    = VGGNet(n_channels=3, n_classes=n_classes).to(DEVICE)
n_params = count_parameters(model)
print(f'Parámetros: {n_params:,} ({n_params/1e6:.3f} M)')

history, best_state = train_model(model, train_loader, val_loader)

# Guardar curvas e historial
save_curves(history, PLOTS_DIR)
with open(METRICS_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

criterion = nn.CrossEntropyLoss()

# Evaluación en validación
model.load_state_dict(best_state['model_state'])
model.to(DEVICE)
_, vl_f1, vl_acc, vl_preds, vl_tgts = run_epoch(
    model, val_loader, criterion, None, DEVICE, training=False)
print(f'[VAL  best] acc={vl_acc:.4f}  f1={vl_f1:.4f}')
save_confusion_matrix(vl_tgts, vl_preds, class_names, PLOTS_DIR / 'confusion_matrix_val.png')
report = classification_report(vl_tgts, vl_preds, target_names=class_names, digits=4, zero_division=0)
(METRICS_DIR / 'classification_report_val.txt').write_text(report)
np.save(METRICS_DIR / 'val_preds.npy',   vl_preds)
np.save(METRICS_DIR / 'val_targets.npy', vl_tgts)

# Evaluación en test
_, ts_f1, ts_acc, ts_preds, ts_tgts = run_epoch(
    model, test_loader, criterion, None, DEVICE, training=False)
print(f'[TEST best] acc={ts_acc:.4f}  f1={ts_f1:.4f}')
(METRICS_DIR / 'test_metrics.txt').write_text(
    f'test_acc={ts_acc:.4f}\ntest_f1={ts_f1:.4f}\n')
np.save(METRICS_DIR / 'test_preds.npy',   ts_preds)
np.save(METRICS_DIR / 'test_targets.npy', ts_tgts)

# Resumen final
summary = (
    f'n_parameters={n_params}\n'
    f'avg_time_per_epoch={np.mean(history["epoch_times"]):.2f}s\n'
    f'best_val_acc={best_state["val_acc"]:.4f}\n'
    f'best_val_f1={max(history["val_f1"]):.4f}\n'
    f'test_acc={ts_acc:.4f}\n'
    f'test_f1={ts_f1:.4f}\n'
)
(METRICS_DIR / 'final_summary.txt').write_text(summary)
print('\n' + summary)

Parámetros: 134,289,477 (134.289 M)
Ep   1/60 | tr_loss=1.6161 tr_f1=0.2102 tr_acc=0.2116 | vl_loss=1.6112 vl_f1=0.1212 vl_acc=0.2057 | lr=0.01000 | 53.1s
  ★ Mejor modelo (val_acc=0.2057)
Ep   2/60 | tr_loss=1.6069 tr_f1=0.2002 tr_acc=0.2106 | vl_loss=1.6086 vl_f1=0.1690 vl_acc=0.2043 | lr=0.01000 | 52.6s
Ep   3/60 | tr_loss=1.6060 tr_f1=0.2087 tr_acc=0.2167 | vl_loss=1.6095 vl_f1=0.1775 vl_acc=0.2077 | lr=0.01000 | 52.7s
  ★ Mejor modelo (val_acc=0.2077)
Ep   4/60 | tr_loss=1.6051 tr_f1=0.2069 tr_acc=0.2169 | vl_loss=1.6080 vl_f1=0.1647 vl_acc=0.2063 | lr=0.01000 | 52.5s
Ep   5/60 | tr_loss=1.6048 tr_f1=0.2086 tr_acc=0.2185 | vl_loss=1.6085 vl_f1=0.1511 vl_acc=0.2020 | lr=0.01000 | 52.6s
Ep   6/60 | tr_loss=1.6042 tr_f1=0.2014 tr_acc=0.2200 | vl_loss=1.6063 vl_f1=0.1369 vl_acc=0.2087 | lr=0.01000 | 52.6s
  ★ Mejor modelo (val_acc=0.2087)
Ep   7/60 | tr_loss=1.6032 tr_f1=0.2131 tr_acc=0.2213 | vl_loss=1.6071 vl_f1=0.1345 vl_acc=0.2020 | lr=0.01000 | 52.6s
Ep   8/60 | tr_loss=1.6028 tr

### V2 — VGG16 + GAP (sin BN)

In [ ]:
# CELDA 3 — Arquitectura VGG16 adaptada a 224x224
import torch
from torch.nn import Conv2d, ReLU, MaxPool2d, Linear, Module, AdaptiveAvgPool2d

class VGGNet(Module):
    def __init__(self, n_channels, n_classes):
        super(VGGNet, self).__init__()
        # Block 1
        self.conv1_1 = Conv2d(n_channels, 64,  kernel_size=3, padding=1, stride=1)
        self.conv1_2 = Conv2d(64,  64,  kernel_size=3, padding=1, stride=1)
        # Block 2
        self.conv2_1 = Conv2d(64,  128, kernel_size=3, padding=1, stride=1)
        self.conv2_2 = Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        # Block 3
        self.conv3_1 = Conv2d(128, 256, kernel_size=3, padding=1, stride=1)
        self.conv3_2 = Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.conv3_3 = Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        # Block 4
        self.conv4_1 = Conv2d(256, 512, kernel_size=3, padding=1, stride=1)
        self.conv4_2 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.conv4_3 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        # Block 5
        self.conv5_1 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.conv5_2 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.conv5_3 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        # Pooling
        self.pool    = MaxPool2d(kernel_size=2, stride=2)
        self.gap  = AdaptiveAvgPool2d((1, 1))  # ← reemplaza las FC + flatten
        self.fc   = Linear(512, n_classes)      # ← solo una FC pequeña
        # Activation
        self.relu = ReLU(inplace=True)

    def forward(self, x):
        # Block 1
        x = self.relu(self.conv1_1(x))
        x = self.relu(self.conv1_2(x))
        x = self.pool(x)
        # Block 2
        x = self.relu(self.conv2_1(x))
        x = self.relu(self.conv2_2(x))
        x = self.pool(x)
        # Block 3
        x = self.relu(self.conv3_1(x))
        x = self.relu(self.conv3_2(x))
        x = self.relu(self.conv3_3(x))
        x = self.pool(x)
        # Block 4
        x = self.relu(self.conv4_1(x))
        x = self.relu(self.conv4_2(x))
        x = self.relu(self.conv4_3(x))
        x = self.pool(x)
        # Block 5
        x = self.relu(self.conv5_1(x))
        x = self.relu(self.conv5_2(x))
        x = self.relu(self.conv5_3(x))
        x = self.pool(x)
        # Fully connected
        x = self.gap(x)         # (batch, 512, 1, 1)
        x = x.flatten(1)        # (batch, 512)
        x = self.fc(x)
        return x

print('Arquitectura definida')

Arquitectura definida


In [ ]:
# CELDA 4 — Imports, configuración, datos y loaders
import os, time, json, copy
from pathlib import Path
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# ── Hiperparámetros ─────────────────────────────────────────
SEED             = 42
EPOCHS           = 60
BATCH_SIZE       = 128
LR               = 0.01      # SGD
MOMENTUM         = 0.9
WEIGHT_DECAY     = 5e-4
NUM_WORKERS      = 4
CHECKPOINT_EVERY = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Rutas ───────────────────────────────────────────────────
DATA_PATH      = '/content/data/Data_subsampled'
OUT_DIR        = Path('/content/drive/MyDrive/Datasets/outputs_vgg16_v2')
CHECKPOINT_DIR = OUT_DIR / 'checkpoints'
PLOTS_DIR      = OUT_DIR / 'plots'
METRICS_DIR    = OUT_DIR / 'metrics'
for p in (CHECKPOINT_DIR, PLOTS_DIR, METRICS_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ── Reproducibilidad ────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ── Media y std del dataset ──────────────────────────────────
def compute_mean_std(data_path):
    ds     = datasets.ImageFolder(data_path, transform=transforms.ToTensor())
    loader = DataLoader(ds, batch_size=128, num_workers=4, shuffle=False,
                        persistent_workers=True)
    mean = torch.zeros(3)
    std  = torch.zeros(3)
    for imgs, _ in loader:
        mean += imgs.mean(dim=[0, 2, 3])
        std  += imgs.std(dim=[0, 2, 3])
    mean /= len(loader)
    std  /= len(loader)
    print(f'Mean: {mean}  Std: {std}')
    return mean.tolist(), std.tolist()

MEAN, STD = compute_mean_std(DATA_PATH)

# ── Transforms ──────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Dataset y splits ────────────────────────────────────────
dataset      = datasets.ImageFolder(root=DATA_PATH, transform=train_transform)
dataset_eval = datasets.ImageFolder(root=DATA_PATH, transform=eval_transform)

class_names = dataset.classes
n_classes   = len(class_names)
print(f'Clases: {class_names}  |  Total: {len(dataset)}  |  Device: {DEVICE}')

targets = np.array(dataset.targets)
indices = np.arange(len(dataset))

train_idx, temp_idx, y_train, y_temp = train_test_split(
    indices, targets, train_size=0.8, stratify=targets, random_state=SEED)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

train_dataset = Subset(dataset,      train_idx)
val_dataset   = Subset(dataset_eval, val_idx)
test_dataset  = Subset(dataset_eval, test_idx)
print(f'train:{len(train_dataset)}  val:{len(val_dataset)}  test:{len(test_dataset)}')

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                 pin_memory=True, persistent_workers=True, prefetch_factor=2)
train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kw)
print('Loaders listos')

Mean: tensor([0.3745, 0.3034, 0.4152])  Std: tensor([0.3549, 0.3974, 0.4199])
Clases: ['Disgust', 'Fear', 'Happy', 'Neutral', 'Sad']  |  Total: 30000  |  Device: cuda
train:24000  val:3000  test:3000
Loaders listos


In [ ]:
# CELDA 5 — Utilidades y funciones de entrenamiento

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def save_checkpoint(state, filename):
    torch.save(state, str(filename))

def save_curves(history, outdir):
    epochs = list(range(1, len(history['train_loss']) + 1))
    for keys, title, fname in [
        (('train_loss','val_loss'), 'Loss',     'loss_curve.png'),
        (('train_f1',  'val_f1'),  'Macro F1', 'f1_curve.png'),
    ]:
        plt.figure()
        for k in keys: plt.plot(epochs, history[k], label=k)
        plt.xlabel('Epoch'); plt.ylabel(title)
        plt.title(f'{title} - Train/Val'); plt.legend(); plt.grid(True)
        plt.savefig(outdir / fname, dpi=150); plt.close()
    import csv
    with open(outdir / 'metrics_per_epoch.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['epoch','train_loss','val_loss','train_f1','val_f1','epoch_time_s'])
        for i in range(len(epochs)):
            w.writerow([epochs[i], history['train_loss'][i], history['val_loss'][i],
                        history['train_f1'][i], history['val_f1'][i], history['epoch_times'][i]])

def save_confusion_matrix(y_true, y_pred, classes, outpath, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    cm_plot = cm.astype('float') / (cm.sum(axis=1)[:, None] + 1e-12) if normalize else cm
    plt.figure(figsize=(8, 6))
    plt.imshow(cm_plot, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de confusión'); plt.colorbar()
    ticks = np.arange(len(classes))
    plt.xticks(ticks, classes, rotation=45, ha='right')
    plt.yticks(ticks, classes)
    fmt = '.2f' if normalize else 'd'
    thresh = cm_plot.max() / 2.
    for i, j in np.ndindex(cm_plot.shape):
        plt.text(j, i, format(cm_plot[i, j], fmt), ha='center',
                 color='white' if cm_plot[i, j] > thresh else 'black')
    plt.ylabel('Clase real'); plt.xlabel('Clase predicha')
    plt.tight_layout(); plt.savefig(outpath, dpi=150); plt.close()
    np.save(str(outpath.with_suffix('.npy')), cm)

def run_epoch(model, loader, criterion, optimizer, device, training=True):
    model.train() if training else model.eval()
    losses, preds_all, tgts_all = [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, tgts in loader:
            imgs, tgts = imgs.to(device), tgts.to(device)
            out  = model(imgs)
            loss = criterion(out, tgts)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            losses.append(loss.item())
            preds_all.extend(out.argmax(1).cpu().numpy())
            tgts_all.extend(tgts.cpu().numpy())
    avg_loss = float(np.mean(losses))
    f1  = float(f1_score(tgts_all, preds_all, average='macro', zero_division=0))
    acc = float(accuracy_score(tgts_all, preds_all))
    return avg_loss, f1, acc, np.array(preds_all), np.array(tgts_all)

def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)

    history = {k: [] for k in ('train_loss','val_loss','train_f1','val_f1','epoch_times')}
    best_val_acc, best_state = -1.0, None

    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()
        tr_loss, tr_f1, tr_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer, DEVICE, training=True)
        vl_loss, vl_f1, vl_acc, vp, vt = run_epoch(model, val_loader, criterion, None, DEVICE, training=False)
        scheduler.step(vl_loss)
        elapsed = time.perf_counter() - t0

        for k, v in zip(('train_loss','val_loss','train_f1','val_f1','epoch_times'),
                        (tr_loss, vl_loss, tr_f1, vl_f1, elapsed)):
            history[k].append(v)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'Ep {epoch:3d}/{EPOCHS} | '
              f'tr_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} tr_acc={tr_acc:.4f} | '
              f'vl_loss={vl_loss:.4f} vl_f1={vl_f1:.4f} vl_acc={vl_acc:.4f} | '
              f'lr={lr_now:.5f} | {elapsed:.1f}s')

        if epoch % CHECKPOINT_EVERY == 0:
            save_checkpoint({'epoch': epoch, 'model_state': model.state_dict(),
                             'optimizer_state': optimizer.state_dict(), 'history': history},
                            CHECKPOINT_DIR / f'checkpoint_epoch_{epoch:03d}.pth')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state = {'epoch': epoch, 'val_acc': vl_acc,
                          'model_state': copy.deepcopy(model.state_dict()),
                          'history': history}
            save_checkpoint(best_state, CHECKPOINT_DIR / 'best_model.pth')
            print(f'  ★ Mejor modelo (val_acc={vl_acc:.4f})')

    save_checkpoint({'epoch': EPOCHS, 'model_state': model.state_dict(), 'history': history},
                    CHECKPOINT_DIR / 'final_model.pth')
    return history, best_state

print('Funciones definidas')

Funciones definidas


In [ ]:
# CELDA 6 — Entrenamiento y evaluación final
model    = VGGNet(n_channels=3, n_classes=n_classes).to(DEVICE)
n_params = count_parameters(model)
print(f'Parámetros: {n_params:,} ({n_params/1e6:.3f} M)')

history, best_state = train_model(model, train_loader, val_loader)

# Guardar curvas e historial
save_curves(history, PLOTS_DIR)
with open(METRICS_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

criterion = nn.CrossEntropyLoss()

# Evaluación en validación
model.load_state_dict(best_state['model_state'])
model.to(DEVICE)
_, vl_f1, vl_acc, vl_preds, vl_tgts = run_epoch(
    model, val_loader, criterion, None, DEVICE, training=False)
print(f'[VAL  best] acc={vl_acc:.4f}  f1={vl_f1:.4f}')
save_confusion_matrix(vl_tgts, vl_preds, class_names, PLOTS_DIR / 'confusion_matrix_val.png')
report = classification_report(vl_tgts, vl_preds, target_names=class_names, digits=4, zero_division=0)
(METRICS_DIR / 'classification_report_val.txt').write_text(report)
np.save(METRICS_DIR / 'val_preds.npy',   vl_preds)
np.save(METRICS_DIR / 'val_targets.npy', vl_tgts)

# Evaluación en test
_, ts_f1, ts_acc, ts_preds, ts_tgts = run_epoch(
    model, test_loader, criterion, None, DEVICE, training=False)
print(f'[TEST best] acc={ts_acc:.4f}  f1={ts_f1:.4f}')
(METRICS_DIR / 'test_metrics.txt').write_text(
    f'test_acc={ts_acc:.4f}\ntest_f1={ts_f1:.4f}\n')
np.save(METRICS_DIR / 'test_preds.npy',   ts_preds)
np.save(METRICS_DIR / 'test_targets.npy', ts_tgts)

# Resumen final
summary = (
    f'n_parameters={n_params}\n'
    f'avg_time_per_epoch={np.mean(history["epoch_times"]):.2f}s\n'
    f'best_val_acc={best_state["val_acc"]:.4f}\n'
    f'best_val_f1={max(history["val_f1"]):.4f}\n'
    f'test_acc={ts_acc:.4f}\n'
    f'test_f1={ts_f1:.4f}\n'
)
(METRICS_DIR / 'final_summary.txt').write_text(summary)
print('\n' + summary)

Parámetros: 14,717,253 (14.717 M)
Ep   1/60 | tr_loss=1.6098 tr_f1=0.1526 tr_acc=0.1975 | vl_loss=1.6097 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 39.7s
  ★ Mejor modelo (val_acc=0.2000)
Ep   2/60 | tr_loss=1.6097 tr_f1=0.1886 tr_acc=0.1961 | vl_loss=1.6095 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 39.4s
Ep   3/60 | tr_loss=1.6097 tr_f1=0.1866 tr_acc=0.1930 | vl_loss=1.6095 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 39.4s
Ep   4/60 | tr_loss=1.6098 tr_f1=0.1841 tr_acc=0.1983 | vl_loss=1.6095 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 39.4s
Ep   5/60 | tr_loss=1.6096 tr_f1=0.1803 tr_acc=0.2002 | vl_loss=1.6095 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 39.4s
Ep   6/60 | tr_loss=1.6097 tr_f1=0.1745 tr_acc=0.1963 | vl_loss=1.6094 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 39.4s
Ep   7/60 | tr_loss=1.6096 tr_f1=0.1921 tr_acc=0.1951 | vl_loss=1.6095 vl_f1=0.0667 vl_acc=0.2000 | lr=0.01000 | 39.4s
Ep   8/60 | tr_loss=1.6096 tr_f1=0.1714 tr_acc=0.1994 | vl_loss=1.6096 vl_f1=0.0667 vl_acc=0.2000 |

### V3 — VGG16 + BN + Dropout

In [ ]:
# CELDA 3 — Arquitectura VGG16 adaptada a 224x224
import torch
from torch.nn import Conv2d, ReLU, MaxPool2d, Linear, Module, Flatten, BatchNorm2d, Dropout

class VGGNet(Module):
    def __init__(self, n_channels, n_classes):
        super(VGGNet, self).__init__()
        # Block 1
        self.conv1_1 = Conv2d(n_channels, 64, kernel_size=3, padding=1)
        self.bn1_1   = BatchNorm2d(64)   # ← NUEVO
        self.conv1_2 = Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn1_2   = BatchNorm2d(64)   # ← NUEVO
        # Block 2
        self.conv2_1 = Conv2d(64,  128, kernel_size=3, padding=1, stride=1)
        self.bn2_1   = BatchNorm2d(128)   # ← NUEVO
        self.conv2_2 = Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn2_2   = BatchNorm2d(128)   # ← NUEVO
        # Block 3
        self.conv3_1 = Conv2d(128, 256, kernel_size=3, padding=1, stride=1)
        self.bn3_1   = BatchNorm2d(256)   # ← NUEVO
        self.conv3_2 = Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn3_2   = BatchNorm2d(256)   # ← NUEVO
        self.conv3_3 = Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn3_3   = BatchNorm2d(256)   # ← NUEVO
        # Block 4
        self.conv4_1 = Conv2d(256, 512, kernel_size=3, padding=1, stride=1)
        self.bn4_1   = BatchNorm2d(512)   # ← NUEVO
        self.conv4_2 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn4_2   = BatchNorm2d(512)   # ← NUEVO
        self.conv4_3 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn4_3   = BatchNorm2d(512)   # ← NUEVO
        # Block 5
        self.conv5_1 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn5_1   = BatchNorm2d(512)   # ← NUEVO
        self.conv5_2 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn5_2   = BatchNorm2d(512)   # ← NUEVO
        self.conv5_3 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn5_3   = BatchNorm2d(512)   # ← NUEVO
        # Pooling
        self.pool    = MaxPool2d(kernel_size=2, stride=2)
        # Flatten
        self.flatten = Flatten()
        # Fully connected  (224 → 5 maxpools → 7x7)
        self.fc1 = Linear(512 * 7 * 7, 4096)
        self.fc2 = Linear(4096, 4096)
        self.fc3 = Linear(4096, n_classes)
        self.drop = Dropout(0.5)
        # Activation
        self.relu = ReLU(inplace=True)

    def forward(self, x):
        # Block 1
        x = self.relu(self.bn1_1(self.conv1_1(x)))  # conv → BN → ReLU
        x = self.relu(self.bn1_2(self.conv1_2(x)))
        x = self.pool(x)
        # Block 2
        x = self.relu(self.bn2_1(self.conv2_1(x)))
        x = self.relu(self.bn2_2(self.conv2_2(x)))
        x = self.pool(x)
        # Block 3
        x = self.relu(self.bn3_1(self.conv3_1(x)))
        x = self.relu(self.bn3_2(self.conv3_2(x)))
        x = self.relu(self.bn3_3(self.conv3_3(x)))
        x = self.pool(x)
        # Block 4
        x = self.relu(self.bn4_1(self.conv4_1(x)))
        x = self.relu(self.bn4_2(self.conv4_2(x)))
        x = self.relu(self.bn4_3(self.conv4_3(x)))
        x = self.pool(x)
        # Block 5
        x = self.relu(self.bn5_1(self.conv5_1(x)))
        x = self.relu(self.bn5_2(self.conv5_2(x)))
        x = self.relu(self.bn5_3(self.conv5_3(x)))
        x = self.pool(x)
        # Flatten
        x = self.flatten(x)
        # Dropout
        x = self.drop(self.relu(self.fc1(x)))  # ← Dropout antes de fc2
        x = self.drop(self.relu(self.fc2(x)))  # ← Dropout antes de fc3
        x = self.fc3(x)
        return x

print('Arquitectura definida')

Arquitectura definida


In [ ]:
# CELDA 4 — Imports, configuración, datos y loaders
import os, time, json, copy
from pathlib import Path
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# ── Hiperparámetros ─────────────────────────────────────────
SEED             = 42
EPOCHS           = 60
BATCH_SIZE       = 128
LR               = 0.01      # SGD
MOMENTUM         = 0.9
WEIGHT_DECAY     = 5e-4
NUM_WORKERS      = 4
CHECKPOINT_EVERY = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Rutas ───────────────────────────────────────────────────
DATA_PATH      = '/content/data/Data_subsampled'
OUT_DIR        = Path('/content/drive/MyDrive/Datasets/outputs_vgg16_v3')
CHECKPOINT_DIR = OUT_DIR / 'checkpoints'
PLOTS_DIR      = OUT_DIR / 'plots'
METRICS_DIR    = OUT_DIR / 'metrics'
for p in (CHECKPOINT_DIR, PLOTS_DIR, METRICS_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ── Reproducibilidad ────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ── Media y std del dataset ──────────────────────────────────
def compute_mean_std(data_path):
    ds     = datasets.ImageFolder(data_path, transform=transforms.ToTensor())
    loader = DataLoader(ds, batch_size=128, num_workers=4, shuffle=False,
                        persistent_workers=True)
    mean = torch.zeros(3)
    std  = torch.zeros(3)
    for imgs, _ in loader:
        mean += imgs.mean(dim=[0, 2, 3])
        std  += imgs.std(dim=[0, 2, 3])
    mean /= len(loader)
    std  /= len(loader)
    print(f'Mean: {mean}  Std: {std}')
    return mean.tolist(), std.tolist()

MEAN, STD = compute_mean_std(DATA_PATH)

# ── Transforms ──────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Dataset y splits ────────────────────────────────────────
dataset      = datasets.ImageFolder(root=DATA_PATH, transform=train_transform)
dataset_eval = datasets.ImageFolder(root=DATA_PATH, transform=eval_transform)

class_names = dataset.classes
n_classes   = len(class_names)
print(f'Clases: {class_names}  |  Total: {len(dataset)}  |  Device: {DEVICE}')

targets = np.array(dataset.targets)
indices = np.arange(len(dataset))

train_idx, temp_idx, y_train, y_temp = train_test_split(
    indices, targets, train_size=0.8, stratify=targets, random_state=SEED)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

train_dataset = Subset(dataset,      train_idx)
val_dataset   = Subset(dataset_eval, val_idx)
test_dataset  = Subset(dataset_eval, test_idx)
print(f'train:{len(train_dataset)}  val:{len(val_dataset)}  test:{len(test_dataset)}')

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                 pin_memory=True, persistent_workers=True, prefetch_factor=2)
train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kw)
print('Loaders listos')

Mean: tensor([0.3745, 0.3034, 0.4152])  Std: tensor([0.3549, 0.3974, 0.4199])
Clases: ['Disgust', 'Fear', 'Happy', 'Neutral', 'Sad']  |  Total: 30000  |  Device: cuda
train:24000  val:3000  test:3000
Loaders listos


In [ ]:
# CELDA 5 — Utilidades y funciones de entrenamiento

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def save_checkpoint(state, filename):
    torch.save(state, str(filename))

def save_curves(history, outdir):
    epochs = list(range(1, len(history['train_loss']) + 1))
    for keys, title, fname in [
        (('train_loss','val_loss'), 'Loss',     'loss_curve.png'),
        (('train_f1',  'val_f1'),  'Macro F1', 'f1_curve.png'),
    ]:
        plt.figure()
        for k in keys: plt.plot(epochs, history[k], label=k)
        plt.xlabel('Epoch'); plt.ylabel(title)
        plt.title(f'{title} - Train/Val'); plt.legend(); plt.grid(True)
        plt.savefig(outdir / fname, dpi=150); plt.close()
    import csv
    with open(outdir / 'metrics_per_epoch.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['epoch','train_loss','val_loss','train_f1','val_f1','epoch_time_s'])
        for i in range(len(epochs)):
            w.writerow([epochs[i], history['train_loss'][i], history['val_loss'][i],
                        history['train_f1'][i], history['val_f1'][i], history['epoch_times'][i]])

def save_confusion_matrix(y_true, y_pred, classes, outpath, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    cm_plot = cm.astype('float') / (cm.sum(axis=1)[:, None] + 1e-12) if normalize else cm
    plt.figure(figsize=(8, 6))
    plt.imshow(cm_plot, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de confusión'); plt.colorbar()
    ticks = np.arange(len(classes))
    plt.xticks(ticks, classes, rotation=45, ha='right')
    plt.yticks(ticks, classes)
    fmt = '.2f' if normalize else 'd'
    thresh = cm_plot.max() / 2.
    for i, j in np.ndindex(cm_plot.shape):
        plt.text(j, i, format(cm_plot[i, j], fmt), ha='center',
                 color='white' if cm_plot[i, j] > thresh else 'black')
    plt.ylabel('Clase real'); plt.xlabel('Clase predicha')
    plt.tight_layout(); plt.savefig(outpath, dpi=150); plt.close()
    np.save(str(outpath.with_suffix('.npy')), cm)

def run_epoch(model, loader, criterion, optimizer, device, training=True):
    model.train() if training else model.eval()
    losses, preds_all, tgts_all = [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, tgts in loader:
            imgs, tgts = imgs.to(device), tgts.to(device)
            out  = model(imgs)
            loss = criterion(out, tgts)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            losses.append(loss.item())
            preds_all.extend(out.argmax(1).cpu().numpy())
            tgts_all.extend(tgts.cpu().numpy())
    avg_loss = float(np.mean(losses))
    f1  = float(f1_score(tgts_all, preds_all, average='macro', zero_division=0))
    acc = float(accuracy_score(tgts_all, preds_all))
    return avg_loss, f1, acc, np.array(preds_all), np.array(tgts_all)

def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)

    history = {k: [] for k in ('train_loss','val_loss','train_f1','val_f1','epoch_times')}
    best_val_acc, best_state = -1.0, None

    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()
        tr_loss, tr_f1, tr_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer, DEVICE, training=True)
        vl_loss, vl_f1, vl_acc, vp, vt = run_epoch(model, val_loader, criterion, None, DEVICE, training=False)
        scheduler.step(vl_loss)
        elapsed = time.perf_counter() - t0

        for k, v in zip(('train_loss','val_loss','train_f1','val_f1','epoch_times'),
                        (tr_loss, vl_loss, tr_f1, vl_f1, elapsed)):
            history[k].append(v)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'Ep {epoch:3d}/{EPOCHS} | '
              f'tr_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} tr_acc={tr_acc:.4f} | '
              f'vl_loss={vl_loss:.4f} vl_f1={vl_f1:.4f} vl_acc={vl_acc:.4f} | '
              f'lr={lr_now:.5f} | {elapsed:.1f}s')

        if epoch % CHECKPOINT_EVERY == 0:
            save_checkpoint({'epoch': epoch, 'model_state': model.state_dict(),
                             'optimizer_state': optimizer.state_dict(), 'history': history},
                            CHECKPOINT_DIR / f'checkpoint_epoch_{epoch:03d}.pth')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state = {'epoch': epoch, 'val_acc': vl_acc,
                          'model_state': copy.deepcopy(model.state_dict()),
                          'history': history}
            save_checkpoint(best_state, CHECKPOINT_DIR / 'best_model.pth')
            print(f'  ★ Mejor modelo (val_acc={vl_acc:.4f})')

    save_checkpoint({'epoch': EPOCHS, 'model_state': model.state_dict(), 'history': history},
                    CHECKPOINT_DIR / 'final_model.pth')
    return history, best_state

print('Funciones definidas')

Funciones definidas


In [ ]:
# CELDA 6 — Entrenamiento y evaluación final
model    = VGGNet(n_channels=3, n_classes=n_classes).to(DEVICE)
n_params = count_parameters(model)
print(f'Parámetros: {n_params:,} ({n_params/1e6:.3f} M)')

history, best_state = train_model(model, train_loader, val_loader)

# Guardar curvas e historial
save_curves(history, PLOTS_DIR)
with open(METRICS_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

criterion = nn.CrossEntropyLoss()

# Evaluación en validación
model.load_state_dict(best_state['model_state'])
model.to(DEVICE)
_, vl_f1, vl_acc, vl_preds, vl_tgts = run_epoch(
    model, val_loader, criterion, None, DEVICE, training=False)
print(f'[VAL  best] acc={vl_acc:.4f}  f1={vl_f1:.4f}')
save_confusion_matrix(vl_tgts, vl_preds, class_names, PLOTS_DIR / 'confusion_matrix_val.png')
report = classification_report(vl_tgts, vl_preds, target_names=class_names, digits=4, zero_division=0)
(METRICS_DIR / 'classification_report_val.txt').write_text(report)
np.save(METRICS_DIR / 'val_preds.npy',   vl_preds)
np.save(METRICS_DIR / 'val_targets.npy', vl_tgts)

# Evaluación en test
_, ts_f1, ts_acc, ts_preds, ts_tgts = run_epoch(
    model, test_loader, criterion, None, DEVICE, training=False)
print(f'[TEST best] acc={ts_acc:.4f}  f1={ts_f1:.4f}')
(METRICS_DIR / 'test_metrics.txt').write_text(
    f'test_acc={ts_acc:.4f}\ntest_f1={ts_f1:.4f}\n')
np.save(METRICS_DIR / 'test_preds.npy',   ts_preds)
np.save(METRICS_DIR / 'test_targets.npy', ts_tgts)

# Resumen final
summary = (
    f'n_parameters={n_params}\n'
    f'avg_time_per_epoch={np.mean(history["epoch_times"]):.2f}s\n'
    f'best_val_acc={best_state["val_acc"]:.4f}\n'
    f'best_val_f1={max(history["val_f1"]):.4f}\n'
    f'test_acc={ts_acc:.4f}\n'
    f'test_f1={ts_f1:.4f}\n'
)
(METRICS_DIR / 'final_summary.txt').write_text(summary)
print('\n' + summary)

Parámetros: 134,289,477 (134.289 M)
Ep   1/60 | tr_loss=1.6451 tr_f1=0.2040 tr_acc=0.2052 | vl_loss=1.6101 vl_f1=0.1509 vl_acc=0.1957 | lr=0.01000 | 52.8s
  ★ Mejor modelo (val_acc=0.1957)
Ep   2/60 | tr_loss=1.6122 tr_f1=0.2022 tr_acc=0.2075 | vl_loss=1.6093 vl_f1=0.1910 vl_acc=0.2063 | lr=0.01000 | 52.6s
  ★ Mejor modelo (val_acc=0.2063)
Ep   3/60 | tr_loss=1.6095 tr_f1=0.2019 tr_acc=0.2127 | vl_loss=1.6089 vl_f1=0.1748 vl_acc=0.2113 | lr=0.01000 | 52.8s
  ★ Mejor modelo (val_acc=0.2113)
Ep   4/60 | tr_loss=1.6080 tr_f1=0.1972 tr_acc=0.2116 | vl_loss=1.6081 vl_f1=0.1637 vl_acc=0.2100 | lr=0.01000 | 52.7s
Ep   5/60 | tr_loss=1.6081 tr_f1=0.1918 tr_acc=0.2114 | vl_loss=1.6086 vl_f1=0.1211 vl_acc=0.2053 | lr=0.01000 | 52.8s
Ep   6/60 | tr_loss=1.6079 tr_f1=0.1794 tr_acc=0.2115 | vl_loss=1.6082 vl_f1=0.1701 vl_acc=0.2180 | lr=0.01000 | 52.7s
  ★ Mejor modelo (val_acc=0.2180)
Ep   7/60 | tr_loss=1.6074 tr_f1=0.1769 tr_acc=0.2135 | vl_loss=1.6079 vl_f1=0.1286 vl_acc=0.2103 | lr=0.01000 | 5

### V4 — VGG16 + BN + GAP + AdamW

In [ ]:
# CELDA 3 — Arquitectura VGG16 adaptada a 224x224
import torch
from torch.nn import Conv2d, ReLU, MaxPool2d, Linear, Module, BatchNorm2d, AdaptiveAvgPool2d

class VGGNet(Module):
    def __init__(self, n_channels, n_classes):
        super(VGGNet, self).__init__()
        # Block 1
        self.conv1_1 = Conv2d(n_channels, 64, kernel_size=3, padding=1)
        self.bn1_1   = BatchNorm2d(64)   # ← NUEVO
        self.conv1_2 = Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn1_2   = BatchNorm2d(64)   # ← NUEVO
        # Block 2
        self.conv2_1 = Conv2d(64,  128, kernel_size=3, padding=1, stride=1)
        self.bn2_1   = BatchNorm2d(128)   # ← NUEVO
        self.conv2_2 = Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn2_2   = BatchNorm2d(128)   # ← NUEVO
        # Block 3
        self.conv3_1 = Conv2d(128, 256, kernel_size=3, padding=1, stride=1)
        self.bn3_1   = BatchNorm2d(256)   # ← NUEVO
        self.conv3_2 = Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn3_2   = BatchNorm2d(256)   # ← NUEVO
        self.conv3_3 = Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn3_3   = BatchNorm2d(256)   # ← NUEVO
        # Block 4
        self.conv4_1 = Conv2d(256, 512, kernel_size=3, padding=1, stride=1)
        self.bn4_1   = BatchNorm2d(512)   # ← NUEVO
        self.conv4_2 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn4_2   = BatchNorm2d(512)   # ← NUEVO
        self.conv4_3 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn4_3   = BatchNorm2d(512)   # ← NUEVO
        # Block 5
        self.conv5_1 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn5_1   = BatchNorm2d(512)   # ← NUEVO
        self.conv5_2 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn5_2   = BatchNorm2d(512)   # ← NUEVO
        self.conv5_3 = Conv2d(512, 512, kernel_size=3, padding=1, stride=1)
        self.bn5_3   = BatchNorm2d(512)   # ← NUEVO
        # Pooling
        self.pool    = MaxPool2d(kernel_size=2, stride=2)
        self.gap  = AdaptiveAvgPool2d((1, 1))  # ← reemplaza las FC + flatten
        self.fc   = Linear(512, n_classes)      # ← solo una FC pequeña
        # Activation
        self.relu = ReLU(inplace=True)

    def forward(self, x):
        # Block 1
        x = self.relu(self.bn1_1(self.conv1_1(x)))  # conv → BN → ReLU
        x = self.relu(self.bn1_2(self.conv1_2(x)))
        x = self.pool(x)
        # Block 2
        x = self.relu(self.bn2_1(self.conv2_1(x)))
        x = self.relu(self.bn2_2(self.conv2_2(x)))
        x = self.pool(x)
        # Block 3
        x = self.relu(self.bn3_1(self.conv3_1(x)))
        x = self.relu(self.bn3_2(self.conv3_2(x)))
        x = self.relu(self.bn3_3(self.conv3_3(x)))
        x = self.pool(x)
        # Block 4
        x = self.relu(self.bn4_1(self.conv4_1(x)))
        x = self.relu(self.bn4_2(self.conv4_2(x)))
        x = self.relu(self.bn4_3(self.conv4_3(x)))
        x = self.pool(x)
        # Block 5
        x = self.relu(self.bn5_1(self.conv5_1(x)))
        x = self.relu(self.bn5_2(self.conv5_2(x)))
        x = self.relu(self.bn5_3(self.conv5_3(x)))
        x = self.pool(x)
        # GAP → FC
        x = self.gap(x)        # (batch, 512, 1, 1)
        x = x.flatten(1)       # (batch, 512)
        return self.fc(x)

print('Arquitectura definida')

Arquitectura definida


In [ ]:
# CELDA 4 — Imports, configuración, datos y loaders
import os, time, json, copy
from pathlib import Path
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# ── Hiperparámetros ─────────────────────────────────────────
SEED             = 42
EPOCHS           = 60
BATCH_SIZE       = 128
LR               = 0.01      # SGD
MOMENTUM         = 0.9
WEIGHT_DECAY     = 5e-4
NUM_WORKERS      = 4
CHECKPOINT_EVERY = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Rutas ───────────────────────────────────────────────────
DATA_PATH      = '/content/data/Data_subsampled'
OUT_DIR        = Path('/content/drive/MyDrive/Datasets/outputs_vgg16_v4')
CHECKPOINT_DIR = OUT_DIR / 'checkpoints'
PLOTS_DIR      = OUT_DIR / 'plots'
METRICS_DIR    = OUT_DIR / 'metrics'
for p in (CHECKPOINT_DIR, PLOTS_DIR, METRICS_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ── Reproducibilidad ────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ── Media y std del dataset ──────────────────────────────────
def compute_mean_std(data_path):
    ds     = datasets.ImageFolder(data_path, transform=transforms.ToTensor())
    loader = DataLoader(ds, batch_size=128, num_workers=4, shuffle=False,
                        persistent_workers=True)
    mean = torch.zeros(3)
    std  = torch.zeros(3)
    for imgs, _ in loader:
        mean += imgs.mean(dim=[0, 2, 3])
        std  += imgs.std(dim=[0, 2, 3])
    mean /= len(loader)
    std  /= len(loader)
    print(f'Mean: {mean}  Std: {std}')
    return mean.tolist(), std.tolist()

MEAN, STD = compute_mean_std(DATA_PATH)

# ── Transforms ──────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Dataset y splits ────────────────────────────────────────
dataset      = datasets.ImageFolder(root=DATA_PATH, transform=train_transform)
dataset_eval = datasets.ImageFolder(root=DATA_PATH, transform=eval_transform)

class_names = dataset.classes
n_classes   = len(class_names)
print(f'Clases: {class_names}  |  Total: {len(dataset)}  |  Device: {DEVICE}')

targets = np.array(dataset.targets)
indices = np.arange(len(dataset))

train_idx, temp_idx, y_train, y_temp = train_test_split(
    indices, targets, train_size=0.8, stratify=targets, random_state=SEED)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

train_dataset = Subset(dataset,      train_idx)
val_dataset   = Subset(dataset_eval, val_idx)
test_dataset  = Subset(dataset_eval, test_idx)
print(f'train:{len(train_dataset)}  val:{len(val_dataset)}  test:{len(test_dataset)}')

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                 pin_memory=True, persistent_workers=True, prefetch_factor=2)
train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kw)
print('Loaders listos')

Mean: tensor([0.3745, 0.3034, 0.4152])  Std: tensor([0.3549, 0.3974, 0.4199])
Clases: ['Disgust', 'Fear', 'Happy', 'Neutral', 'Sad']  |  Total: 30000  |  Device: cuda
train:24000  val:3000  test:3000
Loaders listos


In [ ]:
# CELDA 5 — Utilidades y funciones de entrenamiento

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def save_checkpoint(state, filename):
    torch.save(state, str(filename))

def save_curves(history, outdir):
    epochs = list(range(1, len(history['train_loss']) + 1))
    for keys, title, fname in [
        (('train_loss','val_loss'), 'Loss',     'loss_curve.png'),
        (('train_f1',  'val_f1'),  'Macro F1', 'f1_curve.png'),
    ]:
        plt.figure()
        for k in keys: plt.plot(epochs, history[k], label=k)
        plt.xlabel('Epoch'); plt.ylabel(title)
        plt.title(f'{title} - Train/Val'); plt.legend(); plt.grid(True)
        plt.savefig(outdir / fname, dpi=150); plt.close()
    import csv
    with open(outdir / 'metrics_per_epoch.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['epoch','train_loss','val_loss','train_f1','val_f1','epoch_time_s'])
        for i in range(len(epochs)):
            w.writerow([epochs[i], history['train_loss'][i], history['val_loss'][i],
                        history['train_f1'][i], history['val_f1'][i], history['epoch_times'][i]])

def save_confusion_matrix(y_true, y_pred, classes, outpath, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    cm_plot = cm.astype('float') / (cm.sum(axis=1)[:, None] + 1e-12) if normalize else cm
    plt.figure(figsize=(8, 6))
    plt.imshow(cm_plot, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de confusión'); plt.colorbar()
    ticks = np.arange(len(classes))
    plt.xticks(ticks, classes, rotation=45, ha='right')
    plt.yticks(ticks, classes)
    fmt = '.2f' if normalize else 'd'
    thresh = cm_plot.max() / 2.
    for i, j in np.ndindex(cm_plot.shape):
        plt.text(j, i, format(cm_plot[i, j], fmt), ha='center',
                 color='white' if cm_plot[i, j] > thresh else 'black')
    plt.ylabel('Clase real'); plt.xlabel('Clase predicha')
    plt.tight_layout(); plt.savefig(outpath, dpi=150); plt.close()
    np.save(str(outpath.with_suffix('.npy')), cm)

def run_epoch(model, loader, criterion, optimizer, device, training=True):
    model.train() if training else model.eval()
    losses, preds_all, tgts_all = [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, tgts in loader:
            imgs, tgts = imgs.to(device), tgts.to(device)
            out  = model(imgs)
            loss = criterion(out, tgts)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            losses.append(loss.item())
            preds_all.extend(out.argmax(1).cpu().numpy())
            tgts_all.extend(tgts.cpu().numpy())
    avg_loss = float(np.mean(losses))
    f1  = float(f1_score(tgts_all, preds_all, average='macro', zero_division=0))
    acc = float(accuracy_score(tgts_all, preds_all))
    return avg_loss, f1, acc, np.array(preds_all), np.array(tgts_all)

def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)

    history = {k: [] for k in ('train_loss','val_loss','train_f1','val_f1','epoch_times')}
    best_val_acc, best_state = -1.0, None

    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()
        tr_loss, tr_f1, tr_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer, DEVICE, training=True)
        vl_loss, vl_f1, vl_acc, vp, vt = run_epoch(model, val_loader, criterion, None, DEVICE, training=False)
        scheduler.step(vl_loss)
        elapsed = time.perf_counter() - t0

        for k, v in zip(('train_loss','val_loss','train_f1','val_f1','epoch_times'),
                        (tr_loss, vl_loss, tr_f1, vl_f1, elapsed)):
            history[k].append(v)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'Ep {epoch:3d}/{EPOCHS} | '
              f'tr_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} tr_acc={tr_acc:.4f} | '
              f'vl_loss={vl_loss:.4f} vl_f1={vl_f1:.4f} vl_acc={vl_acc:.4f} | '
              f'lr={lr_now:.5f} | {elapsed:.1f}s')

        if epoch % CHECKPOINT_EVERY == 0:
            save_checkpoint({'epoch': epoch, 'model_state': model.state_dict(),
                             'optimizer_state': optimizer.state_dict(), 'history': history},
                            CHECKPOINT_DIR / f'checkpoint_epoch_{epoch:03d}.pth')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state = {'epoch': epoch, 'val_acc': vl_acc,
                          'model_state': copy.deepcopy(model.state_dict()),
                          'history': history}
            save_checkpoint(best_state, CHECKPOINT_DIR / 'best_model.pth')
            print(f'  ★ Mejor modelo (val_acc={vl_acc:.4f})')

    save_checkpoint({'epoch': EPOCHS, 'model_state': model.state_dict(), 'history': history},
                    CHECKPOINT_DIR / 'final_model.pth')
    return history, best_state

print('Funciones definidas')

Funciones definidas


In [ ]:
# CELDA 6 — Entrenamiento y evaluación final
model    = VGGNet(n_channels=3, n_classes=n_classes).to(DEVICE)
n_params = count_parameters(model)
print(f'Parámetros: {n_params:,} ({n_params/1e6:.3f} M)')

history, best_state = train_model(model, train_loader, val_loader)

# Guardar curvas e historial
save_curves(history, PLOTS_DIR)
with open(METRICS_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

criterion = nn.CrossEntropyLoss()

# Evaluación en validación
model.load_state_dict(best_state['model_state'])
model.to(DEVICE)
_, vl_f1, vl_acc, vl_preds, vl_tgts = run_epoch(
    model, val_loader, criterion, None, DEVICE, training=False)
print(f'[VAL  best] acc={vl_acc:.4f}  f1={vl_f1:.4f}')
save_confusion_matrix(vl_tgts, vl_preds, class_names, PLOTS_DIR / 'confusion_matrix_val.png')
report = classification_report(vl_tgts, vl_preds, target_names=class_names, digits=4, zero_division=0)
(METRICS_DIR / 'classification_report_val.txt').write_text(report)
np.save(METRICS_DIR / 'val_preds.npy',   vl_preds)
np.save(METRICS_DIR / 'val_targets.npy', vl_tgts)

# Evaluación en test
_, ts_f1, ts_acc, ts_preds, ts_tgts = run_epoch(
    model, test_loader, criterion, None, DEVICE, training=False)
print(f'[TEST best] acc={ts_acc:.4f}  f1={ts_f1:.4f}')
(METRICS_DIR / 'test_metrics.txt').write_text(
    f'test_acc={ts_acc:.4f}\ntest_f1={ts_f1:.4f}\n')
np.save(METRICS_DIR / 'test_preds.npy',   ts_preds)
np.save(METRICS_DIR / 'test_targets.npy', ts_tgts)

# Resumen final
summary = (
    f'n_parameters={n_params}\n'
    f'avg_time_per_epoch={np.mean(history["epoch_times"]):.2f}s\n'
    f'best_val_acc={best_state["val_acc"]:.4f}\n'
    f'best_val_f1={max(history["val_f1"]):.4f}\n'
    f'test_acc={ts_acc:.4f}\n'
    f'test_f1={ts_f1:.4f}\n'
)
(METRICS_DIR / 'final_summary.txt').write_text(summary)
print('\n' + summary)

Parámetros: 14,725,701 (14.726 M)
Ep   1/60 | tr_loss=1.6884 tr_f1=0.1945 tr_acc=0.2039 | vl_loss=1.6209 vl_f1=0.1705 vl_acc=0.2027 | lr=0.01000 | 51.4s
  ★ Mejor modelo (val_acc=0.2027)
Ep   2/60 | tr_loss=1.6211 tr_f1=0.1980 tr_acc=0.1991 | vl_loss=1.6101 vl_f1=0.1437 vl_acc=0.2013 | lr=0.01000 | 50.8s
Ep   3/60 | tr_loss=1.6113 tr_f1=0.2034 tr_acc=0.2083 | vl_loss=1.6104 vl_f1=0.1201 vl_acc=0.2037 | lr=0.01000 | 50.8s
  ★ Mejor modelo (val_acc=0.2037)
Ep   4/60 | tr_loss=1.6084 tr_f1=0.2047 tr_acc=0.2136 | vl_loss=1.6103 vl_f1=0.1583 vl_acc=0.1980 | lr=0.01000 | 50.8s
Ep   5/60 | tr_loss=1.6073 tr_f1=0.2073 tr_acc=0.2152 | vl_loss=1.6100 vl_f1=0.1459 vl_acc=0.2020 | lr=0.01000 | 50.9s
Ep   6/60 | tr_loss=1.6075 tr_f1=0.1975 tr_acc=0.2097 | vl_loss=1.6086 vl_f1=0.1413 vl_acc=0.2077 | lr=0.01000 | 50.8s
  ★ Mejor modelo (val_acc=0.2077)
Ep   7/60 | tr_loss=1.6074 tr_f1=0.1946 tr_acc=0.2128 | vl_loss=1.6091 vl_f1=0.1648 vl_acc=0.2003 | lr=0.01000 | 50.9s
Ep   8/60 | tr_loss=1.6062 tr_f